In [1]:
!nvidia-smi

Wed Aug  5 22:09:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.84                 Driver Version: 595.84         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:81:00.0 Off |                  N/A |
|  0%   31C    P8             23W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup Libs & Env

In [ ]:
pip install torchattacks

In [ ]:
pip install timm

In [ ]:
pip install einops

In [ ]:
pip install torchsummary

In [ ]:
pip install scikit-learn

In [2]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

import time
import random
import shutil
import glob
import re

import torchvision
import torchvision.utils
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.amp import autocast, GradScaler

import torchattacks
from torchattacks import PGD, FGSM
from torchsummary import summary
from sklearn.model_selection import train_test_split

In [3]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
pwd

In [4]:
cd Locality-iN-Locality

/data/home/admin/master-deep-learning/Locality-iN-Locality


In [5]:
checkpoint_dir = '../checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Data & Dataloader

### Download the dataset

In [ ]:
!mkdir data

!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip -o data/GTSRB_Final_Training_Images.zip
!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip -o data/GTSRB_Final_Test_Images.zip
!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip -o data/GTSRB_Final_Test_GT.zip

In [ ]:
!unzip data/GTSRB_Final_Training_Images.zip -d data/ > /dev/null 2>&1
!unzip data/GTSRB_Final_Test_Images.zip -d data/ > /dev/null 2>&1
!unzip data/GTSRB_Final_Test_GT.zip -d data/

In [ ]:
data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')


test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)


with open('./data/GT-final_test.csv') as f:
  image_names = f.readlines()


# image_names[0]: is header
for text in image_names[1:]:
  classes = int(text.split(';')[-1])
  image_name = text.split(';')[0]

  test_class_dir = os.path.join(test_dir, f"{classes:04d}")
  os.makedirs(test_class_dir, exist_ok=True)
  image_path = os.path.join(images_dir, image_name)

  shutil.copy(image_path, test_class_dir)

### Create train data

In [6]:
torch.backends.cudnn.benchmark = True

batch_size = 128
num_epochs = 50

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.08),
        ratio=(0.3, 3.3),
        value="random"
    ),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

trainset = torchvision.datasets.ImageFolder(
    root="./data/GTSRB/Final_Training/Images",
    transform=train_transform
)

testset = torchvision.datasets.ImageFolder(
    root="./data/GTSRB/test",
    transform=test_transform
)

train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

test_loader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

# Visualization

In [ ]:
batch = next(iter(train_loader))
train_data = batch[0]

In [ ]:
def normalize_image(image):
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min = image_min, max = image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image


def plot_images(images, labels, classes, normalize=True):
    n_images = len(images)

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(figsize=(20, 20))

    for i in range(rows*cols):

        ax = fig.add_subplot(rows, cols, i+1)

        image = images[i]

        if normalize:
            image = normalize_image(image)

        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')

In [ ]:
classes = trainset.classes
plot_images(batch[0], batch[1], classes)

# LNL Model

In [7]:
from LNL import LNL_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

/home/admin/.conda/envs/gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/admin/.conda/envs/gpu/lib/python3.10/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/admin/.conda/envs/gpu/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/admin/.conda/envs/gpu/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please impor

# Loss - Optimizer - Scheduler

In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=5e-5,
    betas=(0.9,0.999)
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy="cos"
)


# Train

In [9]:
scaler = GradScaler("cuda")

best_loss = float("inf")
best_checkpoint_path = os.path.join(checkpoint_dir, f"best_model.pth")

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.cuda(non_blocking=True)
        labels = labels.cuda(non_blocking=True)

        optimizer.zero_grad()
        with autocast(device_type="cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        scheduler.step()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    lr = scheduler.get_last_lr()[0]

    print("-" * 60)
    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"LR: {lr:.7f}"
    )

    if train_loss < best_loss:
        best_loss = train_loss
        best_checkpoint_path = os.path.join(checkpoint_dir, f"best_model_epoch_{epoch+1}.pth")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }, best_checkpoint_path)
        print(f"Best model saved: {best_checkpoint_path}")

------------------------------------------------------------
Epoch 01/50 | Loss: 3.0095 | Train Acc: 23.76% | LR: 0.0000395
Best model saved: ../checkpoints/best_model_epoch_1.pth
------------------------------------------------------------
Epoch 02/50 | Loss: 1.5190 | Train Acc: 74.19% | LR: 0.0001116
Best model saved: ../checkpoints/best_model_epoch_2.pth
------------------------------------------------------------
Epoch 03/50 | Loss: 0.8180 | Train Acc: 97.82% | LR: 0.0002007
Best model saved: ../checkpoints/best_model_epoch_3.pth
------------------------------------------------------------
Epoch 04/50 | Loss: 0.7532 | Train Acc: 98.91% | LR: 0.0002726
Best model saved: ../checkpoints/best_model_epoch_4.pth
------------------------------------------------------------
Epoch 05/50 | Loss: 0.7349 | Train Acc: 99.10% | LR: 0.0003000
Best model saved: ../checkpoints/best_model_epoch_5.pth
------------------------------------------------------------
Epoch 06/50 | Loss: 0.7250 | Train Acc:

# Test

In [10]:
def evaluate_model(model, test_loader):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * float(correct) / total
    return correct, total, accuracy

### Test the model above

In [ ]:
_, _, acc = evaluate_model(model, test_loader)
print('Standard accuracy: %.2f %%' % acc)

### Test best checkpoint

In [ ]:
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)

print(f"Best model: {best_checkpoint_path}")
checkpoint = torch.load(best_checkpoint_path)
model.load_state_dict(checkpoint["model_state_dict"])

correct, total, _ = evaluate_model(model, test_loader)

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

### Test all checkpoints on checkpoints dir

In [11]:
def test_checkpoints(checkpoint_dir, test_loader, num_classes=43):
    checkpoint_paths = glob.glob(os.path.join(checkpoint_dir, "best_model_epoch_*.pth"))
    checkpoint_paths = sorted(checkpoint_paths,
                              key=lambda x: int(re.search(r"epoch_(\d+)\.pth", x).group(1)),
                              reverse=True)

    results = {}

    decrease_count = 0
    prev_acc = None

    for checkpoint_path in checkpoint_paths:
        if decrease_count == 4:
            break
        
        checkpoint_name = os.path.basename(checkpoint_path)
        print(f"\nTesting: {checkpoint_name}")

        # Tạo model
        test_model = small(pretrained=False)
        test_model.head = nn.Linear(in_features=192, out_features=num_classes, bias=True)
        checkpoint = torch.load(checkpoint_path)
        test_model.load_state_dict(checkpoint["model_state_dict"])

        # Evaluate
        _, _, accuracy = evaluate_model(test_model, test_loader)

        if prev_acc is not None:
            if accuracy < prev_acc:
                decrease_count += 1
        else:
            decrease_count = 0

        prev_acc = accuracy

        results[checkpoint_name] = accuracy
        print(f"Accuracy: {accuracy:.2f}%")

checkpoint_dir = "../checkpoints"
test_checkpoints(checkpoint_dir=checkpoint_dir, test_loader=test_loader, num_classes=43)
# print(results)


Testing: best_model_epoch_47.pth
Accuracy: 99.60%

Testing: best_model_epoch_46.pth
Accuracy: 99.62%

Testing: best_model_epoch_45.pth
Accuracy: 99.60%

Testing: best_model_epoch_43.pth
Accuracy: 99.61%

Testing: best_model_epoch_42.pth
Accuracy: 99.61%

Testing: best_model_epoch_41.pth
Accuracy: 99.43%

Testing: best_model_epoch_40.pth
Accuracy: 99.56%

Testing: best_model_epoch_39.pth
Accuracy: 99.51%

Testing: best_model_epoch_37.pth
Accuracy: 99.24%


In [16]:
top_1_acc = 0

def test_checkpoints(checkpoint_dir, test_loader, num_classes=43):
    checkpoint_paths = glob.glob(os.path.join(checkpoint_dir, "best_model_epoch_*.pth"))
    checkpoint_paths = sorted(checkpoint_paths,
                              key=lambda x: int(re.search(r"epoch_(\d+)\.pth", x).group(1)),
                              reverse=True)

    results = {}

    global top_1_acc

    decrease_count = 0
    prev_acc = None

    for checkpoint_path in checkpoint_paths:
        if decrease_count == 4:
            break
        
        checkpoint_name = os.path.basename(checkpoint_path)
        print(f"\nTesting: {checkpoint_name}", end=' => ')

        # Tạo model
        test_model = small(pretrained=False)
        test_model.head = nn.Linear(in_features=192, out_features=num_classes, bias=True)
        checkpoint = torch.load(checkpoint_path)
        test_model.load_state_dict(checkpoint["model_state_dict"])

        # Evaluate
        _, _, accuracy = evaluate_model(test_model, test_loader)

        if prev_acc is not None:
            if accuracy < prev_acc:
                decrease_count += 1
        else:
            decrease_count = 0

        prev_acc = accuracy

        results[checkpoint_name] = accuracy
        if top_1_acc < accuracy:
            top_1_acc = accuracy

        print(f"Accuracy: {accuracy:.2f}%")

checkpoint_dir = "../checkpoints"
test_checkpoints(checkpoint_dir=checkpoint_dir, test_loader=test_loader, num_classes=43)

print('Standard accuracy: %.2f %%' % top_1_acc)


Testing: best_model_epoch_46.pth => Accuracy: 99.62%
Standard accuracy: 99.62 %
